# SessionRerank+ — Kaggle T4 Execution Notebook
> **v18 — fixed**: argparse in download, torch cache cell split, gemma topk/batch OOM guard, no torchvision
> Run all cells top-to-bottom. GPU T4 required. Total runtime ≈ 30-50 min.

In [ ]:
import subprocess, sys
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NOT FOUND')
print('Python:', sys.version)


In [ ]:
%%capture install_log
import subprocess, sys

# Uninstall conflicting torch first
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchaudio'], check=False)

# Install torch (NO torchvision -- not needed; wastes 2 min)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'setuptools',
    'torch==2.4.1+cu121',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu121',
], check=True)

# Install project deps
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers>=2.7',
    'rank-bm25',
    'transformers>=4.45',
    'huggingface_hub>=0.23',
    'accelerate>=0.30',
    'bitsandbytes>=0.43',
    'scipy>=1.11',
    'scikit-learn>=1.3',
    'numpy<2.1',
], check=True)

print('[install] done')


In [ ]:
# FIX: torch smoke-test in a SEPARATE cell -- old submodules linger in
# sys.modules after uninstall if tested in the same cell as the pip install.
import importlib, sys
to_del = [k for k in sys.modules if k == 'torch' or k.startswith('torch.')]
for k in to_del:
    del sys.modules[k]
importlib.invalidate_caches()

import torch
assert torch.cuda.is_available(), 'CUDA not available -- enable GPU in Kaggle Notebook settings'
print(f'torch {torch.__version__} | CUDA {torch.version.cuda} | {torch.cuda.get_device_name(0)}')


In [ ]:
import os, subprocess, sys

REPO = 'https://github.com/joyjeni/session-aware-toolbench-rerank.git'
WORKDIR = '/kaggle/working/sr'

if os.path.isdir(os.path.join(WORKDIR, '.git')):
    result = subprocess.run(['git', 'pull', '--rebase'], cwd=WORKDIR,
                            capture_output=True, text=True)
    print('git pull:', result.stdout.strip() or result.stderr.strip())
else:
    subprocess.run(['git', 'clone', '--depth=1', REPO, WORKDIR], check=True)
    print('cloned to', WORKDIR)

os.chdir(WORKDIR)
print('cwd:', os.getcwd())


In [ ]:
import subprocess, sys, os
os.chdir('/kaggle/working/sr')

if not os.path.exists('pyproject.toml') and not os.path.exists('setup.py'):
    with open('setup.py', 'w') as f:
        f.write('from setuptools import setup, find_packages\nsetup(name="sessionrerank", version="0.1", packages=find_packages(where="src"), package_dir={"": "src"})\n')
    print('[setup] wrote setup.py')

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'],
                   capture_output=True, text=True)
print(r.stdout[-500:] if r.stdout else '', r.stderr[-200:] if r.returncode else '')
print('[install pkg] done')


In [ ]:
import os

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient  # type: ignore
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    print('[HF_TOKEN] loaded from Kaggle Secrets')
except BaseException:  # FIX: catches AttributeError + Exception
    pass

# hf_token = 'hf_XXXX...'  # <-- uncomment and fill if Secrets are unavailable

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
    print('[HF_TOKEN] set in environment')
else:
    print('[HF_TOKEN] WARNING: not set -- Gemma gated models will fail!')
    print('           Add HF_TOKEN in Notebook -> Add-ons -> Secrets')


In [ ]:
import subprocess, sys, os
os.chdir('/kaggle/working/sr')

data_path = '/kaggle/working/sr/data/toolbench_hf'
if os.path.isdir(data_path) and len(os.listdir(data_path)) > 2:
    print('[download] data already present at', data_path)
else:
    print('[download] fetching ToolBench from HuggingFace...')
    # FIX: removed --out arg; download.py v18 uses argparse parse_known_args
    # ROOT is auto-computed from __file__, points to /kaggle/working/sr/data/toolbench_hf
    r = subprocess.run(
        [sys.executable, '-m', 'sessionrerank.data.download'],
        cwd='/kaggle/working/sr', check=True,
    )
    print('[download] complete')

import pathlib
files = list(pathlib.Path(data_path).rglob('*.parquet'))
print(f'found {len(files)} parquet files')
for f in sorted(files)[:8]:
    print(' ', f.relative_to('/kaggle/working/sr'))


In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/sr/src')
from sessionrerank.data.loader import build_corpus, load_eval_split

corpus = build_corpus()
print(f'corpus: {len(corpus)} APIs')
qs = load_eval_split('G1_instruction')
print(f'G1_instruction eval queries: {len(qs)}')
print(f'first query: {qs[0].query[:80]}')


In [ ]:
import subprocess, sys, os, time
os.chdir('/kaggle/working/sr')

cmd = [
    sys.executable, '-m', 'sessionrerank.eval.run_all',
    '--device', 'cuda',
    '--encoder', 'sentence-transformers/all-MiniLM-L6-v2',
    '--seeds', '0,1,2',
    '--fit-sessions', '4000',
    '--cooc',
    '--cooc-sessions', '20000',
    '--cross-encoder',
    '--cross-encoder-model', 'cross-encoder/ms-marco-MiniLM-L-6-v2',
    '--cross-encoder-topk', '100',
    '--gemma-rerank', 'google/gemma-2-2b-it',
    # FIX: topk=50 + batch-size=4 prevents T4 CUDA OOM (gemma-2-2b-it in 4bit ~2.2GB)
    '--gemma-rerank-topk', '50',
    '--gemma-rerank-batch-size', '4',
    '--gemma-4bit',
    '--free-embed-after-encode',
    '--resume',
    '--out', 'results/metrics.json',
]

print('[run_all] starting... (~30-50 min on T4)')
t0 = time.time()
r = subprocess.run(cmd, cwd='/kaggle/working/sr')
print(f'[run_all] exit={r.returncode}  elapsed={time.time()-t0:.0f}s')


In [ ]:
import json, pathlib

result_path = pathlib.Path('/kaggle/working/sr/results/metrics.json')
if not result_path.exists():
    result_path = pathlib.Path('/kaggle/working/sr/results/metrics.json.partial')
    print('[results] showing partial checkpoint')

if result_path.exists():
    data = json.loads(result_path.read_text())
    splits = data.get('splits', {})
    if not splits and 'seeds' in data:
        splits = data['seeds'][0].get('splits', {}) if data['seeds'] else {}
    print(f"{'Split':<20} {'Regime':<10} {'Method':<52} {'NDCG@5':>8} {'Hit@5':>8}")
    print('-'*104)
    for split, sblk in splits.items():
        for regime, rblk in sblk.get('regimes', {}).items():
            for method, metrics in rblk.items():
                if isinstance(metrics, dict) and 'NDCG@5' in metrics:
                    v = metrics['NDCG@5']
                    h = metrics['Hit@5']
                    nd = v['mean'] if isinstance(v, dict) else v
                    hi = h['mean'] if isinstance(h, dict) else h
                    print(f'{split:<20} {regime:<10} {method:<52} {nd:>8.4f} {hi:>8.4f}')
else:
    print('[results] no results file found')
